In [0]:

# If needed on a fresh cluster, run once:
%pip install geopandas openpyxl matplotlib shapely scikit-learn xlsxwriter mapclassify

# ================================================================
# WILD BIRD MORTALITY — OPTION A (MAPS + CSVs)
# Uses ReportSpecies only; allows 3 species: bird of prey, corvid, song bird
# ================================================================

import os
import re
import calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------------------------
# CONFIG
# -------------------------
INPUT_XLSX = '/Volumes/prd_dash_lab/aphw_wbm_unrestricted/wild_bird_mortality/WildBirdMortality_010423_110925_UKFC_EJ_LIMS.xlsx'
COUNTIES_SHP_DIR = '/dbfs/mnt/base/unrestricted/source_ons_open_geography_portal/dataset_counties_uni_authorities_bdy_uk_bfe/format_SHP_counties_uni_authorities_bdy_uk_bfe/LATEST_counties_uni_authorities_bdy_uk_bfe/'
OUTPUT_PATH = '/Volumes/prd_dash_lab/aphw_wbm_unrestricted/wild_bird_mortality/WNV/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Target species (OPTION A) — corrected to match lower-casing later
VALID_SPECIES = ['bird of prey', 'corvid', 'song bird']

# Colours/markers for plots (species)
SPECIES_COLORS = {
    'bird of prey': '#1a9850',
    'corvid': '#d73027',
    'song bird': '#4575b4'
}
SPECIES_MARKERS = {
    'bird of prey': 's',
    'corvid': 'o',
    'song bird': '^'
}

In [0]:
# -------------------------
# LOAD + CLEAN BASE DATA
# -------------------------
df = pd.read_excel(INPUT_XLSX, engine='openpyxl')
df.columns = [str(c).strip() for c in df.columns]

# Clean relevant string fields
for c in ['ReportSpecies','ReportMonth','UKFC_Collected',
          'UKFC_NoPickupReason','UKFC_NoCollectionReason',
          'deadbird_reportedlocation','SubmittedCountry']:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip().replace({"": pd.NA})

# Numerics
for c in ['ReportYear','ReportBirdCount','UKFCBirdCount','Dlong','Dlat']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Month normalisation (keeps May–Sep only)
MONTH_SYNONYMS = {"sept":"September","jul":"July","aug":"August","jun":"June","may":"May"}

def normalize_month(m):
    s = str(m).strip()
    if s == "" or s.lower() in {"nan","none","<na>"}:
        return np.nan
    try:
        i = int(float(s))
        if 1 <= i <= 12:
            return calendar.month_name[i]
    except Exception:
        pass
    low = s.lower()
    if low in MONTH_SYNONYMS:
        return MONTH_SYNONYMS[low]
    return s.title()

df['ReportMonth'] = df.get('ReportMonth', pd.Series(dtype="string")).map(normalize_month)
MONTHS_LIST = ["May","June","July","August","September"]
df = df[df['ReportMonth'].isin(MONTHS_LIST)].copy()

In [0]:
# -------------------------
# RULES 1–4 (as per your pipeline)
# -------------------------

# Rule 1 — public estimate
df['public_reported_count'] = df['ReportBirdCount']

# Rule 2 — collection event
ukfc_yes = df.get('UKFC_Collected','').str.lower().eq('yes')
ukfc_count_pos = df.get('UKFCBirdCount',0).fillna(0) > 0
df['is_collection_event'] = ukfc_yes & ukfc_count_pos
df['analysis_count_confirmed'] = np.where(df['is_collection_event'], df['UKFCBirdCount'], np.nan)
df['is_unverified_public_estimate'] = ~df['is_collection_event']

# Rule 4 — tie-break using no-collection reasons
NON_COLLECTION_HINTS = [
    'swabs taken','duplicate report','no carcases at location specified',
    'collector not available','failed collection (carcass gone)',
    'dangerous location','already collected from area (positive - waiting 2 weeks)',
    'inaccessible location'
]
rp = df.get('UKFC_NoPickupReason','').str.lower()
rc = df.get('UKFC_NoCollectionReason','').str.lower()
has_hint = rp.isin(NON_COLLECTION_HINTS) | rc.isin(NON_COLLECTION_HINTS)

df['is_collection_event'] = np.where(
    has_hint & (df['UKFCBirdCount'].fillna(0) <= 0),
    False,
    df['is_collection_event']
)
df['is_unverified_public_estimate'] = ~df['is_collection_event']
df['analysis_count_confirmed'] = np.where(df['is_collection_event'], df['UKFCBirdCount'], np.nan)

# -------------------------
# OPTION A — STRICT SPECIES FILTERING FROM ReportSpecies
# -------------------------
df['species_category'] = (
    df['ReportSpecies']
      .astype("string")
      .str.strip()
      .str.lower()
)
# Keep only the 3 desired species
df = df[df['species_category'].isin(VALID_SPECIES)].copy()

# Drop rows without coordinates
df = df.dropna(subset=['Dlong','Dlat'])

# -------------------------
# GEODATAFRAME + COUNTY ASSIGNMENT
# -------------------------
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Dlong'], df['Dlat']),
    crs="EPSG:4326"
)
counties_gdf = gpd.read_file(COUNTIES_SHP_DIR)

# Reproject if needed
if gdf.crs != counties_gdf.crs:
    gdf = gdf.to_crs(counties_gdf.crs)

# Robust county name detection (prefers CTYUA/LAD latest code)
preferred = [c for c in counties_gdf.columns if re.match(r"(CTYUA|LAD)\d{2}NM$", c)]
if preferred:
    county_col = sorted(preferred, reverse=True)[0]
else:
    fallback = [c for c in counties_gdf.columns if ('NM' in c) or ('name' in c.lower())]
    county_col = fallback[0] if fallback else None
if not county_col:
    raise ValueError("Could not detect a county name column in the shapefile.")

# Spatial join: assign county
result = gpd.sjoin(
    gdf,
    counties_gdf[['geometry', county_col]],
    how='left',
    predicate='intersects'
).rename(columns={county_col: 'county'})

# -------------------------
# LAND MASK (keep boundary points)
# -------------------------
land_poly = counties_gdf.dissolve().geometry.union_all()
pre_land = result.copy()
on_land_mask = pre_land['county'].notna() & pre_land.geometry.covered_by(land_poly)
result = pre_land[on_land_mask].copy()

In [0]:
# -------------------------
# OFFSHORE SUMMARIES (by Year–Month and by Year)
# -------------------------
offshore = pre_land[~on_land_mask].copy()

def safe_sum(series):
    s = series.dropna()
    if s.empty: return 0
    return s.sum()

# Build species flags for the 3 allowed species (for offshore aggregation)
offshore_flags = {}
for sp in VALID_SPECIES:
    key = sp.replace(" ", "_")
    offshore_flags[f'is_{key}'] = (offshore['species_category'] == sp).astype(int)
offshore = offshore.assign(**offshore_flags)

offshore_agg = {
    'records_offshore': ('county','size'),
    'public_reported_sum_offshore': ('public_reported_count', safe_sum),
    'confirmed_collected_sum_offshore': ('analysis_count_confirmed', safe_sum)
}
for sp in VALID_SPECIES:
    key = sp.replace(" ", "_")
    offshore_agg[f'{key}_reports_offshore'] = (f'is_{key}', 'sum')

offshore_summary_by_year_month = (
    offshore.groupby(['ReportYear','ReportMonth'])
    .agg(**offshore_agg)
    .reset_index()
    .sort_values(['ReportYear','ReportMonth'])
)
offshore_summary_by_year = (
    offshore.groupby(['ReportYear'])
    .agg(**offshore_agg)
    .reset_index()
    .sort_values(['ReportYear'])
)

offshore_summary_by_year_month.to_csv(f"{OUTPUT_PATH}/offshore_summary_by_year_month.csv", index=False)
offshore_summary_by_year.to_csv(f"{OUTPUT_PATH}/offshore_summary_by_year.csv", index=False)

# -------------------------
# AGGREGATIONS & CSVs (county counts + county summary)
# -------------------------
# County x Month rollups with species detail (for display & QC)
base = (
    result.groupby(['ReportYear','ReportMonth','county'])
    .agg(report_count=('county','size'),
         public_reported_sum=('public_reported_count', safe_sum),
         confirmed_collected_sum=('analysis_count_confirmed', safe_sum))
    .reset_index()
)

# Species‑specific monthly counts (restricted to VALID_SPECIES)
from functools import reduce
species_monthlys = []
for sp in VALID_SPECIES:
    key = sp.replace(" ", "_")
    res_sp = result.assign(**{
        f'is_{key}': (result['species_category'] == sp).astype(int),
        f'public_reported_count__{key}': np.where(result['species_category'].eq(sp), result['public_reported_count'], np.nan),
        f'confirmed_count__{key}': np.where(result['species_category'].eq(sp), result['analysis_count_confirmed'], np.nan)
    })
    sm = (
        res_sp.groupby(['ReportMonth','county'])
        .agg(**{
            f'{sp} reports': (f'is_{key}', 'sum'),
            f'{sp} public_sum': (f'public_reported_count__{key}', safe_sum),
            f'{sp} confirmed_sum': (f'confirmed_count__{key}', safe_sum),
        })
        .reset_index()
    )
    species_monthlys.append(sm)

county_counts_by_month = reduce(
    lambda left, right: left.merge(right, on=['ReportMonth','county'], how='left'),
    [base] + species_monthlys
)
county_counts_by_month.to_csv(f"{OUTPUT_PATH}/county_counts_by_month.csv", index=False)

#A lambda function is a small anonymous function. It can take any number of arguments, but can only have one expression
# County display table (base totals)
county_all_months_base = (
    result.groupby('county')
    .agg(report_count=('county','size'),
         public_reported_sum=('public_reported_count', safe_sum),
         confirmed_collected_sum=('analysis_count_confirmed', safe_sum))
    .reset_index()
    .rename(columns={'county':'County'})
)
county_all_months_base.to_csv(f"{OUTPUT_PATH}/county_counts_display.csv", index=False)

# County summary across ALL months (with species breakdown)
species_totals = []
for sp in VALID_SPECIES:
    key = sp.replace(" ", "_")
    tmp = result.assign(
        **{
            f'is_{key}': (result['species_category'] == sp).astype(int),
            f'public_reported_count__{key}': np.where(result['species_category'].eq(sp), result['public_reported_count'], np.nan),
            f'confirmed_count__{key}': np.where(result['species_category'].eq(sp), result['analysis_count_confirmed'], np.nan)
        }
    )
    g = (
        tmp.groupby('county')
        .agg(**{
            f'{sp} Reports': (f'is_{key}', 'sum'),
            f'{sp} Public Sum (Unverified)': (f'public_reported_count__{key}', safe_sum),
            f'{sp} Confirmed Sum': (f'confirmed_count__{key}', safe_sum),
        })
        .reset_index()
        .rename(columns={'county': 'County'})
    )
    species_totals.append(g)

county_summary_all_months = reduce(
    lambda left, right: left.merge(right, on='County', how='left'),
    [county_all_months_base] + species_totals
).sort_values(by=['confirmed_collected_sum','public_reported_sum'], ascending=[False, False])

county_summary_all_months.to_csv(f"{OUTPUT_PATH}/county_summary_all_months.csv", index=False)

In [0]:
# -------------------------
# PLOTTING HELPERS (consistent titles & legends)
# -------------------------

def apply_wbm_title(fig, subtitle):
    """
    Main title at the absolute top; subtitle (month / description) directly under it.
    """
    fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
    fig.text(0.5, 0.94, subtitle, ha='center', va='top', fontsize=16)

def add_map_key_inset(ax, species_list, status_mode=None, box=(0.02, 0.02, 0.30, 0.26)):
    """
    Draws a map key inset box on the given axes.
    - species_list: list of species present (match VALID_SPECIES)
    - status_mode: None (both statuses), 'confirmed' (only confirmed), or 'public' (only public)
    - box: (x, y, w, h) in axes-relative coordinates
    """
    inset = ax.inset_axes(box)
    inset.axis('off')

    # Species handles: filled marker in species color with black edge
    species_handles = [
        Line2D([0], [0],
               marker=SPECIES_MARKERS[s],
               color=SPECIES_COLORS[s],
               markerfacecolor=SPECIES_COLORS[s],
               markeredgecolor='black',
               markersize=9, linewidth=0, label=s.title())
        for s in species_list
    ]

    # Status handles
    status_handles_all = [
        Line2D([0],[0], marker='o', color='black',
               markerfacecolor='black', markeredgecolor='black',
               markersize=8, linewidth=0, label='Confirmed UKFC collection'),
        Line2D([0],[0], marker='o', color='black',
               markerfacecolor='white', markeredgecolor='black',
               markersize=7, linewidth=1.2, label='Public report (unverified)')
    ]
    if status_mode == 'confirmed':
        status_handles = status_handles_all[:1]  # only confirmed
    elif status_mode == 'public':
        status_handles = status_handles_all[1:]  # only public
    else:
        status_handles = status_handles_all

    handles = species_handles + status_handles
    if handles:
        leg = inset.legend(handles=handles, loc='upper left',
                           title='Map Key', frameon=True,
                           fontsize=9, title_fontsize=10)
        # Make the box slightly translucent
        leg.get_frame().set_alpha(0.9)

def choropleth_by_county(counties_gdf, county_col, values_df, value_col, subtitle, outpath,
                         cmap='Blues', scheme=None, vmin=None, vmax=None, legend_label=None,
                         fill_na_with_zero=True):
    fig, ax = plt.subplots(figsize=(18, 18), constrained_layout=True)
    if 'county' not in values_df.columns:
        if 'County' in values_df.columns:
            values_df = values_df.rename(columns={'County': 'county'})
        else:
            raise ValueError("values_df must contain a 'county' column.")
    merged = counties_gdf.merge(values_df, left_on=county_col, right_on='county', how='left')
    if fill_na_with_zero and value_col in merged.columns:
        merged[value_col] = merged[value_col].fillna(0)

    legend_kw = {"label": legend_label or value_col.replace('_',' ').title(), "shrink": 0.6}
    merged.plot(
        column=value_col, cmap=cmap, legend=True, ax=ax,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        scheme=scheme, k=5 if scheme else None,
        vmin=vmin, vmax=vmax,
        legend_kwds=legend_kw
    )
    apply_wbm_title(fig, subtitle)
    ax.axis('off')
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def overlay_points(counties_gdf, subset_gdf, subtitle, outpath,
                   differentiate_confirmed=True, status_filter=None,
                   inset_status=True):
    """
    Plots species points over county boundaries.
    - differentiate_confirmed: if True and status is mixed, draw confirmed vs public differently
    - status_filter: None | 'confirmed' | 'public' to export status-only maps
    - inset_status: add a map key inset with species + status
    """
    fig, ax = plt.subplots(figsize=(18, 18), constrained_layout=True)
    counties_gdf.plot(ax=ax, color='white', edgecolor='lightgrey', linewidth=0.6)

    # Keep on-land only (same land mask logic as earlier)
    land_poly = counties_gdf.dissolve().geometry.union_all()
    subset = subset_gdf[subset_gdf.geometry.covered_by(land_poly)].copy()

    # Optional status filtering for status-only exports
    status_mode = None
    if status_filter in {'confirmed', 'public'}:
        status_mode = status_filter
        if status_filter == 'confirmed':
            subset = subset[subset['is_collection_event'] == True]
            differentiate_confirmed = False
        else:
            subset = subset[subset['is_collection_event'] == False]
            differentiate_confirmed = False

    present_species = [sp for sp in VALID_SPECIES if sp in subset['species_category'].unique()]

    # Plot per-species layers
    for sp in present_species:
        layer_all = subset[subset['species_category'] == sp]
        if layer_all.empty:
            continue

        if differentiate_confirmed and 'is_collection_event' in layer_all.columns:
            layer_conf = layer_all[layer_all['is_collection_event'] == True]
            layer_pub  = layer_all[layer_all['is_collection_event'] == False]

            # Confirmed: filled marker with black edge
            if not layer_conf.empty:
                layer_conf.plot(
                    ax=ax,
                    color=SPECIES_COLORS[sp],           # face color
                    edgecolor='black',
                    marker=SPECIES_MARKERS[sp],
                    markersize=28,
                    linewidth=0.5
                )

            # Public: hollow style (white face, coloured edge)
            if not layer_pub.empty:
                layer_pub.plot(
                    ax=ax,
                    color='white',                      # "hollow" look
                    edgecolor=SPECIES_COLORS[sp],
                    marker=SPECIES_MARKERS[sp],
                    markersize=18,
                    linewidth=1.2
                )
        else:
            # Single-status or fallback
            layer_all.plot(
                ax=ax,
                color=SPECIES_COLORS[sp],
                edgecolor='black',
                marker=SPECIES_MARKERS[sp],
                markersize=22,
                linewidth=0.5
            )

    apply_wbm_title(fig, subtitle)
    ax.axis('off')

    # Map key inset (species + status)
    if inset_status and present_species:
        add_map_key_inset(ax, present_species, status_mode=status_mode)

    fig.savefig(outpath, dpi=250)
    plt.close(fig)

In [0]:
# -------------------------
# PLOTTING — FULL SET
# -------------------------
if result.empty:
    raise ValueError("No data after filters (species/months/coordinates/land-only). Check inputs.")

In [0]:
# (1) Monthly county counts — choropleths
for month in sorted(result['ReportMonth'].dropna().unique()):
    monthly_counts = (
        result[result['ReportMonth'] == month]['county']
        .value_counts().rename_axis('county').reset_index(name='Count')
    )
    choropleth_by_county(
        counties_gdf, county_col, monthly_counts, 'Count',
        f"All Reports by County — {month}",
        f"{OUTPUT_PATH}/county_counts_{month}.png",
        scheme=None
    )


In [0]:
# (2) Combined county counts (All months May–Sep)
combined_counts = (
    result['county'].value_counts().rename_axis('county').reset_index(name='Count')
)
choropleth_by_county(
    counties_gdf, county_col, combined_counts, 'Count',
    "All Reports by County — May to September",
    f"{OUTPUT_PATH}/combined_county_counts.png",
    scheme=None
)


In [0]:
# (3) Combined county counts BY YEAR — choropleths
years = sorted(result['ReportYear'].dropna().unique())
for year in years:
    subset = result[result['ReportYear'] == year]
    if subset.empty:
        continue
    yearly_counts = (
        subset['county'].value_counts().rename_axis('county').reset_index(name='Count')
    )
    choropleth_by_county(
        counties_gdf, county_col, yearly_counts, 'Count',
        f"All Reports by County — {int(year)}",
        f"{OUTPUT_PATH}/combined_county_counts_{int(year)}.png",
        scheme=None
    )

In [0]:
# (4) ALL SPECIES — PUBLIC vs CONFIRMED BY YEAR — choropleths (unchanged)
for year in years:
    subset = result[result['ReportYear'] == year]
    if subset.empty:
        continue
    for metric, label, col in [
        ('public', 'Public Reports (Unverified)', 'public_reported_count'),
        ('confirmed', 'Confirmed Carcasses', 'analysis_count_confirmed')
    ]:
        county_values = (
            subset.groupby('county').agg(Value=(col, safe_sum)).reset_index()
        )
        choropleth_by_county(
            counties_gdf, county_col, county_values, 'Value',
            f"{label} by County — {int(year)}",
            f"{OUTPUT_PATH}/county_all_species_{metric}_{int(year)}.png",
            scheme=None, legend_label=label
        )


In [0]:
# (5) Per‑year point map overlay — MIXED + STATUS-ONLY
for year in years:
    subset = result[result['ReportYear'] == year]
    if subset.empty: continue

    # Mixed (confirmed + public, differentiated)
    overlay_points(
        counties_gdf, subset,
        f"Species Overlay — {int(year)}",
        f"{OUTPUT_PATH}/all_three_species_combined_{int(year)}.png",
        differentiate_confirmed=True, status_filter=None
    )
    # Confirmed-only
    overlay_points(
        counties_gdf, subset,
        f"Species Overlay — {int(year)} (Confirmed only)",
        f"{OUTPUT_PATH}/all_three_species_confirmed_only_{int(year)}.png",
        differentiate_confirmed=False, status_filter='confirmed'
    )
    # Public-only
    overlay_points(
        counties_gdf, subset,
        f"Species Overlay — {int(year)} (Public only)",
        f"{OUTPUT_PATH}/all_three_species_public_only_{int(year)}.png",
        differentiate_confirmed=False, status_filter='public'
    )

In [0]:
# (6) Points by species (categorical, all reports May–Sep, all years together)
#     -> keep as a simple mixed overview with inset key
overlay_points(
    counties_gdf, result,
    "Species Distribution (All Reports) — May to September",
    f"{OUTPUT_PATH}/combined_species_distribution.png",
    differentiate_confirmed=True, status_filter=None
)


In [0]:
# (7) One figure: all species overlaid (all months May–Sep) — MIXED + STATUS-ONLY
overlay_points(
    counties_gdf, result,
    "Species Overlay — May to September",
    f"{OUTPUT_PATH}/all_three_species_combined_map.png",
    differentiate_confirmed=True, status_filter=None
)
overlay_points(
    counties_gdf, result,
    "Species Overlay — May to September (Confirmed only)",
    f"{OUTPUT_PATH}/all_three_species_confirmed_only_map.png",
    differentiate_confirmed=False, status_filter='confirmed'
)
overlay_points(
    counties_gdf, result,
    "Species Overlay — May to September (Public only)",
    f"{OUTPUT_PATH}/all_three_species_public_only_map.png",
    differentiate_confirmed=False, status_filter='public'
)

In [0]:
# (8) Monthly 5‑panel combining species (May–Sep, ALL years merged)
#     A) Mixed (differentiated) with inset on first panel
fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
for i, m in enumerate(MONTHS_LIST):
    ax_i = axs[i]
    month_data = result[result['ReportMonth'] == m]
    counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
    for sp in VALID_SPECIES:
        sp_month = month_data[month_data['species_category'] == sp]
        if sp_month.empty: continue
        sp_conf = sp_month[sp_month['is_collection_event'] == True]
        sp_pub  = sp_month[sp_month['is_collection_event'] == False]
        if not sp_conf.empty:
            sp_conf.plot(ax=ax_i,
                         color=SPECIES_COLORS[sp], edgecolor='black',
                         marker=SPECIES_MARKERS[sp], markersize=24, linewidth=0.5)
        if not sp_pub.empty:
            sp_pub.plot(ax=ax_i,
                        color='white', edgecolor=SPECIES_COLORS[sp],
                        marker=SPECIES_MARKERS[sp], markersize=14, linewidth=1.1)
    ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')

fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
fig.text(0.5, 0.94, "Combined Species Distribution — All Years (May–September)", ha='center', va='top', fontsize=16)
present_species_all = [s for s in VALID_SPECIES if s in result['species_category'].unique()]
if len(axs) > 0 and present_species_all:
    add_map_key_inset(axs[0], present_species_all, status_mode=None)
fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly.png", dpi=200)
plt.close(fig)


In [0]:


# (8) Monthly 5‑panel combining species (May–Sep, ALL years merged)
#     A) Mixed (differentiated) with inset on first panel
fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
for i, m in enumerate(MONTHS_LIST):
    ax_i = axs[i]
    month_data = result[result['ReportMonth'] == m]
    counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
    for sp in VALID_SPECIES:
        sp_month = month_data[month_data['species_category'] == sp]
        if sp_month.empty: continue
        sp_conf = sp_month[sp_month['is_collection_event'] == True]
        sp_pub  = sp_month[sp_month['is_collection_event'] == False]
        if not sp_conf.empty:
            sp_conf.plot(ax=ax_i,
                         color=SPECIES_COLORS[sp], edgecolor='black',
                         marker=SPECIES_MARKERS[sp], markersize=24, linewidth=0.5)
        if not sp_pub.empty:
            sp_pub.plot(ax=ax_i,
                        color='white', edgecolor=SPECIES_COLORS[sp],
                        marker=SPECIES_MARKERS[sp], markersize=14, linewidth=1.1)
    ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')

fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
fig.text(0.5, 0.94, "Combined Species Distribution — All Years (May–September)", ha='center', va='top', fontsize=16)
present_species_all = [s for s in VALID_SPECIES if s in result['species_category'].unique()]
if len(axs) > 0 and present_species_all:
    add_map_key_inset(axs[0], present_species_all, status_mode=None)
fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly.png", dpi=200)
plt.close(fig)

#     B) Confirmed-only panel
fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
for i, m in enumerate(MONTHS_LIST):
    ax_i = axs[i]
    month_data = result[(result['ReportMonth'] == m) & (result['is_collection_event'] == True)]
    counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
    for sp in VALID_SPECIES:
        layer = month_data[month_data['species_category'] == sp]
        if layer.empty: continue
        layer.plot(ax=ax_i,
                   color=SPECIES_COLORS[sp], edgecolor='black',
                   marker=SPECIES_MARKERS[sp], markersize=24, linewidth=0.5)
    ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')

fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
fig.text(0.5, 0.94, "Confirmed Collections Only — All Years (May–September)", ha='center', va='top', fontsize=16)
if len(axs) > 0 and present_species_all:
    add_map_key_inset(axs[0], present_species_all, status_mode='confirmed')
fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly_confirmed_only.png", dpi=200)
plt.close(fig)

#     C) Public-only panel
fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
for i, m in enumerate(MONTHS_LIST):
    ax_i = axs[i]
    month_data = result[(result['ReportMonth'] == m) & (result['is_collection_event'] == False)]
    counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
    for sp in VALID_SPECIES:
        layer = month_data[month_data['species_category'] == sp]
        if layer.empty: continue
        layer.plot(ax=ax_i,
                   color='white', edgecolor=SPECIES_COLORS[sp],
                   marker=SPECIES_MARKERS[sp], markersize=16, linewidth=1.1)
    ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')

fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
fig.text(0.5, 0.94, "Public Reports Only — All Years (May–September)", ha='center', va='top', fontsize=16)
if len(axs) > 0 and present_species_all:
    add_map_key_inset(axs[0], present_species_all, status_mode='public')
fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly_public_only.png", dpi=200)
plt.close(fig)

In [0]:

# (9) Yearly 5‑panel monthly maps (May–Sep) with species overlay — MIXED + STATUS-ONLY
for year in years:
    year_data = result[result['ReportYear'] == year]
    if year_data.empty: continue
    species_year = [s for s in VALID_SPECIES if s in year_data['species_category'].unique()]

    #   A) Mixed
    fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
    for i, m in enumerate(MONTHS_LIST):
        ax_i = axs[i]
        counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
        month_data = year_data[year_data['ReportMonth'] == m]
        for sp in VALID_SPECIES:
            sp_month = month_data[month_data['species_category'] == sp]
            if sp_month.empty: continue
            sp_conf = sp_month[sp_month['is_collection_event'] == True]
            sp_pub  = sp_month[sp_month['is_collection_event'] == False]
            if not sp_conf.empty:
                sp_conf.plot(ax=ax_i,
                             color=SPECIES_COLORS[sp], edgecolor='black',
                             marker=SPECIES_MARKERS[sp], markersize=24, linewidth=0.5)
            if not sp_pub.empty:
                sp_pub.plot(ax=ax_i,
                            color='white', edgecolor=SPECIES_COLORS[sp],
                            marker=SPECIES_MARKERS[sp], markersize=14, linewidth=1.1)
        ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')
    fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
    fig.text(0.5, 0.94, f"Combined Species Distribution — {int(year)}", ha='center', va='top', fontsize=16)
    if len(axs) > 0 and species_year:
        add_map_key_inset(axs[0], species_year, status_mode=None)
    fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly_{int(year)}.png", dpi=200)
    plt.close(fig)

    #   B) Confirmed-only
    fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
    for i, m in enumerate(MONTHS_LIST):
        ax_i = axs[i]
        counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
        month_data = year_data[(year_data['ReportMonth'] == m) & (year_data['is_collection_event'] == True)]
        for sp in VALID_SPECIES:
            layer = month_data[month_data['species_category'] == sp]
            if layer.empty: continue
            layer.plot(ax=ax_i,
                       color=SPECIES_COLORS[sp], edgecolor='black',
                       marker=SPECIES_MARKERS[sp], markersize=24, linewidth=0.5)
        ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')
    fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
    fig.text(0.5, 0.94, f"Confirmed Collections Only — {int(year)}", ha='center', va='top', fontsize=16)
    if len(axs) > 0 and species_year:
        add_map_key_inset(axs[0], species_year, status_mode='confirmed')
    fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly_confirmed_only_{int(year)}.png", dpi=200)
    plt.close(fig)

    #   C) Public-only
    fig, axs = plt.subplots(1, len(MONTHS_LIST), figsize=(40, 10), constrained_layout=True)
    for i, m in enumerate(MONTHS_LIST):
        ax_i = axs[i]
        counties_gdf.plot(ax=ax_i, color='white', edgecolor='lightgrey')
        month_data = year_data[(year_data['ReportMonth'] == m) & (year_data['is_collection_event'] == False)]
        for sp in VALID_SPECIES:
            layer = month_data[month_data['species_category'] == sp]
            if layer.empty: continue
            layer.plot(ax=ax_i,
                       color='white', edgecolor=SPECIES_COLORS[sp],
                       marker=SPECIES_MARKERS[sp], markersize=16, linewidth=1.1)
        ax_i.set_title(f"{m}", fontsize=12); ax_i.axis('off')
    fig.suptitle("Wild Bird Mortality", fontsize=22, fontweight='bold', y=0.98)
    fig.text(0.5, 0.94, f"Public Reports Only — {int(year)}", ha='center', va='top', fontsize=16)
    if len(axs) > 0 and species_year:
        add_map_key_inset(axs[0], species_year, status_mode='public')
    fig.savefig(f"{OUTPUT_PATH}/combined_species_monthly_public_only_{int(year)}.png", dpi=200)
    plt.close(fig)

In [0]:
print(f"[INFO] All outputs written to: {OUTPUT_PATH}")
for fn in [
    "offshore_summary_by_year_month.csv",
    "offshore_summary_by_year.csv",
    "county_counts_by_month.csv",
    "county_counts_display.csv",
    "county_summary_all_months.csv",
    "combined_county_counts.png",
    "combined_species_distribution.png",
    "all_three_species_combined_map.png",
    "all_three_species_confirmed_only_map.png",
    "all_three_species_public_only_map.png",
]:
    print(" -", os.path.join(OUTPUT_PATH, fn))